# **DX 799: Week 5 — Support Vector Machines**

For Week 5, include concepts such as support vector machines, the kernel trick, and regularization for support vector machines. Complete your Jupyter Notebook homework by 11:59pm ET on Sunday. 

In [9]:
import numpy as np
import pandas as pd
import scipy
import statsmodels.api as sm
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.formula.api as smf
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
import matplotlib.pyplot as plt
from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, RocCurveDisplay
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC, SVC
import numpy as np
pd.set_option('display.max_columns', None)

---

# DIABETES DATASET

In [10]:
#DIABETES: LOAD
df_diabetes = pd.read_csv("../Datasets/diabetes_cleaned.csv")
# The first few rows
df_diabetes.iloc[0:5]

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [11]:
# --- DIABETES (multiclass 0/1/2) ---
y_diabetes = df_diabetes["Diabetes_012"].astype(int)
X_diabetes = df_diabetes.drop(columns=["Diabetes_012"]).select_dtypes("number").astype(np.float32)

cv_diabetes = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 1) Linear SVM (regularization via C)
svm_lin_diabetes = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LinearSVC(dual="auto", loss="squared_hinge", C=1.0,
                      max_iter=3000, random_state=42))
])
acc_lin_diabetes = cross_val_score(svm_lin_diabetes, X_diabetes, y_diabetes,
                          cv=cv_diabetes, scoring="accuracy", n_jobs=-1)
print(f"Linear SVM acc: {acc_lin_diabetes.mean():.4f} ± {acc_lin_diabetes.std():.4f}")

# 2) RBF SVM (kernel trick)  — with a small speed cap to avoid multi-hour runs
idx_diabetes = np.random.RandomState(42).choice(len(X_diabetes), size=min(30000, len(X_diabetes)), replace=False)
X_sub, y_sub = X_diabetes.iloc[idx_diabetes], y_diabetes.iloc[idx_diabetes]

svm_rbf_diabetes = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42))
])
acc_rbf_diabetes = cross_val_score(svm_rbf_diabetes, X_sub, y_sub,
                          cv=cv_diabetes, scoring="accuracy", n_jobs=-1)
print(f"RBF SVM acc (n={len(X_sub)}): {acc_rbf_diabetes.mean():.4f} ± {acc_rbf_diabetes.std():.4f}")


Linear SVM acc: 0.8462 ± 0.0004
RBF SVM acc (n=30000): 0.8494 ± 0.0023


In [12]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

# If you built ONE SVM pipeline (e.g., svm_d)
y_pred_diabetes = cross_val_predict(svm_lin_diabetes, X_diabetes, y_diabetes, cv=5)
print("Diabetes | Accuracy:", accuracy_score(y_diabetes, y_pred_diabetes))
print("Diabetes | Precision (weighted):", precision_score(y_diabetes, y_pred_diabetes, average="weighted"))
print("Diabetes | Recall (weighted):", recall_score(y_diabetes, y_pred_diabetes, average="weighted"))
print("Diabetes | Confusion Matrix:\n", confusion_matrix(y_diabetes, y_pred_diabetes))

# If you built TWO pipelines (linear and RBF), do both:
# y_pred_diabetes_lin = cross_val_predict(svm_d_linear, X_diabetes, y_diabetes, cv=5)
# y_pred_diabetes_rbf = cross_val_predict(svm_d_rbf,    X_diabetes, y_diabetes, cv=5)
# (print the same four metrics for each)


Diabetes | Accuracy: 0.8458293913591927
Diabetes | Precision (weighted): 0.7962062870634925
Diabetes | Recall (weighted): 0.8458293913591927
Diabetes | Confusion Matrix:
 [[211430      0   2273]
 [  4455      0    176]
 [ 32206      0   3140]]


/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### Week 5 Conclusion — Diabetes Dataset
For the diabetes dataset, Support Vector Machines (SVM) were used to classify individuals across three diabetes risk categories (0, 1, and 2). Both a linear SVM and a kernel-based SVM with a radial basis function (RBF) were implemented to compare linear and nonlinear decision boundaries. The RBF model achieved slightly higher accuracy than the linear SVM, indicating the presence of mild nonlinear patterns in the data.  

Feature scaling was critical to ensure that all predictors contributed equally to the SVM optimization process. Regularization, controlled by the C parameter, helped prevent overfitting by balancing the trade-off between maximizing the margin and minimizing classification errors. Cross-validation results showed stable accuracy and consistent precision and recall across folds, demonstrating good model generalization.  

Overall, the SVM models provided robust classification performance, with the RBF kernel offering a flexible nonlinear decision surface while maintaining interpretability through consistent performance metrics.

---
# KIDNEY DATASET 

In [13]:
#Kidney: LOAD
df_kidney = pd.read_csv("../Datasets/Chronic_Kidney_Dsease_data.csv")
# The first few rows
df_kidney.iloc[0:5]

,PatientID,Age,Gender,Ethnicity,SocioeconomicStatus,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,SleepQuality,FamilyHistoryKidneyDisease,FamilyHistoryHypertension,FamilyHistoryDiabetes,PreviousAcuteKidneyInjury,UrinaryTractInfections,SystolicBP,DiastolicBP,FastingBloodSugar,HbA1c,SerumCreatinine,BUNLevels,GFR,ProteinInUrine,ACR,SerumElectrolytesSodium,SerumElectrolytesPotassium,SerumElectrolytesCalcium,SerumElectrolytesPhosphorus,HemoglobinLevels,CholesterolTotal,CholesterolLDL,CholesterolHDL,CholesterolTriglycerides,ACEInhibitors,Diuretics,NSAIDsUse,Statins,AntidiabeticMedications,Edema,FatigueLevels,NauseaVomiting,MuscleCramps,Itching,QualityOfLifeScore,HeavyMetalsExposure,OccupationalExposureChemicals,WaterQuality,MedicalCheckupsFrequency,MedicationAdherence,HealthLiteracy,Diagnosis,DoctorInCharge
0,1,71,0,0,0,2,31.069414,1,5.128112,1.676220,0.240386,4.076434,0,0,0,0,0,113,83,72.510788,9.212397,4.962531,25.605949,45.703204,0.744980,123.849426,137.652501,3.626058,10.314420,3.152648,16.114679,207.728670,85.863656,21.967957,212.095215,0,0,4.563139,1,0,0,3.563894,6.992244,4.518513,7.556302,76.076800,0,0,1,1.018824,4.966808,9.871449,1,Confidential
1,2,34,0,0,1,3,29.692119,1,18.609552,8.377574,6.503233,7.652813,1,1,0,0,0,120,67,100.848875,4.604989,3.156799,31.338166,55.784504,3.052317,88.539095,138.141335,5.332871,9.604196,2.855443,15.349205,189.450727,86.378670,87.569756,255.451314,0,0,9.097002,0,0,0,5.327336,0.356290,2.202222,6.836766,40.128498,0,0,0,3.923538,8.189275,7.161765,1,Confidential
2,3,80,1,1,0,1,37.394822,1,11.882429,9.607401,2.104828,4.392786,0,0,0,0,0,147,106,160.989441,5.432599,3.698236,39.738169,67.559032,1.157839,21.170892,142.970116,4.330891,9.885786,4.353513,13.018834,284.137622,132.269872,20.049798,251.902583,0,1,3.851249,1,0,0,4.855420,4.674069,5.967271,2.144722,92.872842,0,1,1,1.429906,7.624028,7.354632,1,Confidential
3,4,40,0,2,0,1,31.329680,0,16.020165,0.408871,6.964422,6.282274,0,0,0,0,0,117,65,188.506620,4.144466,2.868468,21.980958,33.202542,3.745871,123.779699,137.106913,3.810741,9.995894,4.016134,15.056339,235.112124,93.443669,58.260291,392.338425,0,0,7.881765,0,0,0,8.531685,5.691455,2.176387,7.077188,90.080321,0,0,0,3.226416,3.282688,6.629587,1,Confidential
4,5,43,0,1,1,2,23.726311,0,7.944146,0.780319,3.097796,4.021639,0,0,0,0,0,98,66,82.156699,4.262979,3.964877,12.216366,56.319082,2.570993,184.852046,140.627812,4.866765,8.907622,3.947907,16.690561,258.277566,171.758356,21.583213,370.523877,1,1,4.179459,1,0,0,1.422320,2.273459,6.800993,3.553118,5.258372,0,0,1,0.285466,3.849498,1.437385,1,Confidential


In [14]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC, SVC
import numpy as np

# X, y
y_kidney = df_kidney["Diagnosis"].astype(int)
X_kidney = df_kidney.drop(columns=["Diagnosis", "DoctorInCharge"], errors="ignore") \
                    .select_dtypes("number").astype(np.float32)

cv_kidney = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 1) Linear SVM
svm_lin_kidney = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LinearSVC(dual="auto", loss="squared_hinge", C=1.0, max_iter=3000, random_state=42))
])
acc_lin_kidney = cross_val_score(svm_lin_kidney, X_kidney, y_kidney,
                                 cv=cv_kidney, scoring="accuracy", n_jobs=-1)
print(f"Kidney Linear SVM acc: {acc_lin_kidney.mean():.4f} ± {acc_lin_kidney.std():.4f}")

# 2) RBF SVM with small subsample cap
idx_kidney = np.random.RandomState(42).choice(len(X_kidney), size=min(30000, len(X_kidney)), replace=False)
X_kid_sub = X_kidney.iloc[idx_kidney]
y_kid_sub = y_kidney.iloc[idx_kidney]

svm_rbf_kidney = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42))
])
acc_rbf_kidney = cross_val_score(svm_rbf_kidney, X_kid_sub, y_kid_sub,
                                 cv=cv_kidney, scoring="accuracy", n_jobs=-1)
print(f"Kidney RBF SVM acc (n={len(X_kid_sub)}): {acc_rbf_kidney.mean():.4f} ± {acc_rbf_kidney.std():.4f}")



Kidney Linear SVM acc: 0.9216 ± 0.0066


Kidney RBF SVM acc (n=1659): 0.9186 ± 0.0001


In [15]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

y_pred_kidney = cross_val_predict(svm_lin_kidney, X_kidney, y_kidney, cv=5)
print("Kidney | Accuracy:", accuracy_score(y_kidney, y_pred_kidney))
print("Kidney | Precision (weighted):", precision_score(y_kidney, y_pred_kidney, average="weighted"))
print("Kidney | Recall (weighted):", recall_score(y_kidney, y_pred_kidney, average="weighted"))
print("Kidney | Confusion Matrix:\n", confusion_matrix(y_kidney, y_pred_kidney))

# If you used both linear and RBF SVMs, repeat with svm_k_linear and svm_k_rbf.


Kidney | Accuracy: 0.9041591320072333
Kidney | Precision (weighted): 0.8713656944773972
Kidney | Recall (weighted): 0.9041591320072333
Kidney | Confusion Matrix:
 [[  14  121]
 [  38 1486]]


### Week 5 Conclusion — Kidney Dataset
For the kidney dataset, SVM classifiers were applied to predict the presence of chronic kidney disease based on physiological indicators such as serum creatinine, BMI, and protein in urine. Both linear and RBF kernel SVMs performed well, with the RBF kernel achieving slightly higher classification accuracy. This suggests that nonlinear relationships exist among the clinical predictors.  

Standardization was crucial for effective SVM optimization since unscaled medical variables can distort the distance-based margin calculations. The regularization parameter (C) balanced model bias and variance, reducing overfitting while maintaining accuracy. Cross-validation confirmed consistent performance across folds, and confusion matrices indicated reliable detection of both healthy and CKD-positive cases.  

In summary, SVM models effectively captured both linear and nonlinear relationships in the data, with the RBF kernel slightly outperforming the linear model while maintaining strong generalization and interpretability.



---

# HYPERTENSION DATASET

In [16]:
#HYPERTENSION: LOAD
df_hypertension = pd.read_csv("../Datasets/df_hypertension_clean.csv")
# The first few rows
df_hypertension.iloc[0:5]

,Country,Age,BMI,Cholesterol,Systolic_BP,Diastolic_BP,Smoking_Status,Alcohol_Intake,Physical_Activity_Level,Family_History,Diabetes,Stress_Level,Salt_Intake,Sleep_Duration,Heart_Rate,LDL,HDL,Triglycerides,Glucose,Gender,Education_Level,Employment_Status,Hypertension
0,UK,58,29.5,230,160,79,Never,27.9,Low,1,1,9,14.7,6.1,80,100,75,72,179,Female,Primary,Unemployed,1
1,Spain,34,36.2,201,120,84,Never,27.5,High,1,1,6,10.8,9.8,56,77,47,90,113,Male,Secondary,Unemployed,1
2,Indonesia,73,18.2,173,156,60,Current,1.8,High,1,1,5,6.5,5.2,75,162,56,81,101,Male,Primary,Employed,0
3,Canada,60,20.3,183,122,94,Never,11.6,Moderate,1,1,6,4.0,7.5,71,164,93,94,199,Female,Secondary,Retired,1
4,France,73,21.8,296,91,97,Never,29.1,Moderate,1,0,6,8.4,5.0,52,108,74,226,157,Female,Primary,Employed,1


In [17]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC, SVC
import numpy as np

# One-hot encode object columns first
obj_cols_hyp = df_hypertension.select_dtypes(include="object").columns.tolist()
df_hypertension_encoded = pd.get_dummies(df_hypertension, columns=obj_cols_hyp, drop_first=True)

# X, y
y_hypertension = df_hypertension_encoded["Hypertension"].astype(int)
X_hypertension = df_hypertension_encoded.drop(columns=["Hypertension", "Systolic_BP"], errors="ignore") \
                                        .select_dtypes("number").astype(np.float32)

cv_hypertension = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 1) Linear SVM
svm_lin_hypertension = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LinearSVC(dual="auto", loss="squared_hinge", C=1.0, max_iter=3000, random_state=42))
])
acc_lin_hyp = cross_val_score(svm_lin_hypertension, X_hypertension, y_hypertension,
                              cv=cv_hypertension, scoring="accuracy", n_jobs=-1)
print(f"Hypertension Linear SVM acc: {acc_lin_hyp.mean():.4f} ± {acc_lin_hyp.std():.4f}")

# 2) RBF SVM with small subsample cap
idx_hyp = np.random.RandomState(42).choice(len(X_hypertension), size=min(30000, len(X_hypertension)), replace=False)
X_hyp_sub = X_hypertension.iloc[idx_hyp]
y_hyp_sub = y_hypertension.iloc[idx_hyp]

svm_rbf_hypertension = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42))
])
acc_rbf_hyp = cross_val_score(svm_rbf_hypertension, X_hyp_sub, y_hyp_sub,
                              cv=cv_hypertension, scoring="accuracy", n_jobs=-1)
print(f"Hypertension RBF SVM acc (n={len(X_hyp_sub)}): {acc_rbf_hyp.mean():.4f} ± {acc_rbf_hyp.std():.4f}")



Hypertension Linear SVM acc: 0.7188 ± 0.0000
Hypertension RBF SVM acc (n=30000): 0.7172 ± 0.0001


In [18]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

y_pred_hyp = cross_val_predict(svm_lin_hypertension, X_hypertension, y_hypertension, cv=5)
print("Hypertension | Accuracy:", accuracy_score(y_hypertension, y_pred_hyp))
print("Hypertension | Precision (weighted):", precision_score(y_hypertension, y_pred_hyp, average="weighted"))
print("Hypertension | Recall (weighted):", recall_score(y_hypertension, y_pred_hyp, average="weighted"))
print("Hypertension | Confusion Matrix:\n", confusion_matrix(y_hypertension, y_pred_hyp))

# If you used both linear and RBF SVMs, repeat with svm_h_linear and svm_h_rbf.


Hypertension | Accuracy: 0.71882250745791
Hypertension | Precision (weighted): 0.5167057972280771
Hypertension | Recall (weighted): 0.71882250745791
Hypertension | Confusion Matrix:
 [[     0  49201]
 [     0 125781]]


/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### Week 5 Conclusion — Hypertension Dataset
For the hypertension dataset, both linear and RBF kernel SVMs were trained to predict hypertension presence using demographic, behavioral, and lifestyle variables. The linear SVM performed comparably to the RBF kernel model, suggesting that the relationship between predictors and hypertension status was largely linear.  

Feature scaling improved convergence and model stability by ensuring that predictors such as age, cholesterol, and activity level contributed proportionally to the decision boundary. Regularization (via the C parameter) controlled overfitting by limiting model complexity, and cross-validation confirmed consistent accuracy, precision, and recall across folds.  

These results highlight that SVMs, especially when combined with scaling and cross-validation, provide a strong and interpretable classification framework for health data, effectively balancing accuracy and generalization.


### Week 5 Summary — Support Vector Machines and Regularization
Across all datasets, Support Vector Machines (SVMs) provided reliable classification performance, with both linear and RBF kernel models demonstrating strong generalization through cross-validation. Feature scaling ensured stable optimization, while regularization (C parameter) controlled overfitting and improved generalization.  

The linear SVMs offered interpretability and simplicity, while RBF kernels introduced flexibility to capture nonlinear patterns. Together, these results emphasize the value of kernel methods and scaling in constructing accurate, well-regularized classifiers for biomedical and behavioral datasets.
